# PE scaling and its effects on spatial structure and robustness

This notebook aims to help you reproduce the results found in subsection 5.5 (Linking Index-Based Spatial Organization to Robustness via Positional Scaling). In this notebook, you will load our checkpoints (check repo README if you haven't downloaded them yet), run a sweep of magnitudes from values in the range [0.1, 1.0], and evaluate robustness and SSDC under RPI for each magnitude.

**Memory note** : each model condition should be loaded and evaluated in a fresh session.
Running multiple conditions sequentially will OOM on most free-tier cloud hardware.

In [ ]:
# Importing needed dependencies

import sys
import os
sys.path.append(os.path.abspath("../"))
from src.main.model import VisionTransformer, ViTConfig
from src.metrics.robustness import evaluate_robustness_jpeg, evaluate_robustness_gaussian_blur
from src.metrics.ssdc import evaluate_ssdc
import torch


# Downloading Imagenet-100 dataset

from datasets import load_dataset

dataset_val = load_dataset("clane9/imagenet-100", split="validation")

In [ ]:
# Set which model you would like to test

condition = "APE" # Change to: "APE", "RoPE", "SPE", "RPT", or "ablated"
model_num = 1 # Must be an integer from 1 to 4

# Initializing model

DEFAULT_CONFIG = ViTConfig()
ViT = VisionTransformer(DEFAULT_CONFIG, condition)

# Loading checkpoint
path = os.path.abspath("../../models") + f"{condition}{model_num}_imagenet_100_60"
ViT.load_state_dict(torch.load(path)["model"])

## Important note

Due to the memory-heavy nature of this experiment, we will divide the code into two cells. For users without access to high-memory hardware, the cell directly under this will evaluate SSDC under RPI and robustness for one magnitude value that you can change by hand. If memory isn't a worry for you, the second cell under this will inlude code that will allow you to do a full sweep of all magnitude values tested in the paper in one go.

**Note on visualization**: Unlike the previous notebook, this experiment is summarized primarily through scalar metrics (ΔSSDC and robustness scores). Visualizing token cosine similarity matrices at every magnitude value is possible, but adds little additional insight once the spatial patterns introduced in robustness_ssdc_and_rpi.ipynb are understood.

In [ ]:
magnitude = 1.0 # Change to 0.1, 0.2, 0.3, ... , 1.0

# Evaluate fragility scores of the model to two different forms of corruption

jpeg_score = evaluate_robustness_jpeg(ViT, dataset_val, RPI = False, magnitude)
gaussian_blur_score = evaluate_robustness_gaussian_blur(ViT, dataset_val, RPI = False, magnitude)

print(f"Fragility Score of model with PE magnitude {magnitude} under JPEG corruption: {jpeg_score}")
print(f"Fragility Score of model with PE magnitude {magnitude} under Gaussian Blur: {gaussian_blur_score}")

ssdc_scores, cosine_maps = evaluate_ssdc(ViT, dataset_val, RPI = True, magnitude)

delta_ssdc = ssdc_scores[1] - ssdc_scores[0]
print(f"Delta SSDC under RPI with PE magnitude {magnitude}: {delta_ssdc}")

In [ ]:
# One shot the magnitude scaling experiment

results = []
for magnitude in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:
    # Evaluate fragility scores of the model to two different forms of corruption

    jpeg_score = evaluate_robustness_jpeg(ViT, dataset_val, RPI = False, magnitude)
    gaussian_blur_score = evaluate_robustness_gaussian_blur(ViT, dataset_val, RPI = False, magnitude)

    print(f"Fragility Score of model with PE magnitude {magnitude} under JPEG corruption: {jpeg_score}")
    print(f"Fragility Score of model with PE magnitude {magnitude} under Gaussian Blur: {gaussian_blur_score}")

    ssdc_scores, cosine_maps = evaluate_ssdc(ViT, dataset_val, RPI = True, magnitude)

    delta_ssdc = ssdc_scores[1] - ssdc_scores[0]
    print(f"Delta SSDC under RPI with PE magnitude {magnitude}: {delta_ssdc}")